In [8]:
import math

In [9]:
file = open("input3.txt", "r")
data = file.read().splitlines()
file.close()

nucleotides_pool = data[0].split(",")
target_sequence = data[1]
sid = list(map(int, data[2].split()))
weights = sid[len(sid) - len(target_sequence):]

In [10]:
def utility_function(gene, target_gene, weight):
    utility_score = 0
    
    n = max(len(gene), len(target_gene))
    while len(gene) < n:
        gene += "-"
    while len(target_gene) < n:
        target_gene += "-"
    while len(weight) < n:
        weight.append(1)
    
    gene_ascii = []
    target_ascii = []
    for i in range(n):
        if gene[i] == "-":
            gene_ascii.append(0)
        else:
            gene_ascii.append(ord(gene[i]))
        if target_gene[i] == "-":
            target_ascii.append(0)
        else:
            target_ascii.append(ord(target_gene[i]))

    for i in range(n):
        utility_score += weight[i] * abs(gene_ascii[i] - target_ascii[i])
    return (-1) * utility_score

In [11]:
def minimax_search(gene, nucleotides_pool, is_max, target_sequence, alpha, beta, weight):
    if nucleotides_pool == []:
        return gene, utility_function(gene, target_sequence, weight)
    
    best_sequence = ""
    
    if is_max:
        local_value = -math.inf
        for i in range(len(nucleotides_pool)):
            new_nucleotides_pool = nucleotides_pool.copy()
            new_nucleotide = new_nucleotides_pool.pop(i)
            new_gene = gene + new_nucleotide
            sequence, value = minimax_search(new_gene, new_nucleotides_pool, False, target_sequence, alpha, beta, weight)
            if value > local_value:
                local_value = value
                best_sequence = sequence
            alpha = max(alpha, local_value)
            if alpha >= beta:
                break
    else:
        local_value = math.inf
        for i in range(len(nucleotides_pool)):
            new_nucleotides_pool = nucleotides_pool.copy()
            new_nucleotide = new_nucleotides_pool.pop(i)
            new_gene = gene + new_nucleotide
            sequence, value = minimax_search(new_gene, new_nucleotides_pool, True, target_sequence, alpha, beta, weight)
            if value < local_value:
                local_value = value
                best_sequence = sequence
            beta = min(beta, local_value)
            if alpha >= beta:
                break

    return best_sequence, local_value

In [12]:
best_gene, best_utility = minimax_search("", nucleotides_pool, True, target_sequence, -math.inf, math.inf, weights)
print("Best Gene Sequence:", best_gene)
print("Best Utility Score:", best_utility)

Best Gene Sequence: AGCT
Best Utility Score: -54


In [13]:
modified_nucleotides_pool = nucleotides_pool.copy()
modified_nucleotides_pool.append("S")

booster_value = int(data[2][:1]+data[2][2:3]) / 100

In [14]:
def modified_minimax_search(gene, nucleotides_pool, is_max, target_sequence, alpha, beta, weight):
    if nucleotides_pool == []:
        return gene, utility_function(gene, target_sequence, weight)
    
    best_sequence = ""
    
    if is_max:
        local_value = -math.inf
        for i in range(len(nucleotides_pool)):
            new_nucleotides_pool = nucleotides_pool.copy()
            new_nucleotide = new_nucleotides_pool.pop(i)
            new_gene = gene + new_nucleotide
            new_weight = weight.copy()
            if "S" in new_gene:
                s_index = 0
                for j in range(len(new_gene)):
                    if new_gene[j] == "S":
                        s_index = j
                        break
                for k in range(s_index, len(new_weight)):
                    new_weight[k] = new_weight[k] * booster_value
            sequence, value = minimax_search(new_gene, new_nucleotides_pool, False, target_sequence, alpha, beta, new_weight)
            if value > local_value:
                local_value = value
                best_sequence = sequence
            alpha = max(alpha, local_value)
            if alpha >= beta:
                break
    else:
        local_value = math.inf
        for i in range(len(nucleotides_pool)):
            new_nucleotides_pool = nucleotides_pool.copy()
            new_nucleotide = new_nucleotides_pool.pop(i)
            new_gene = gene + new_nucleotide
            sequence, value = minimax_search(new_gene, new_nucleotides_pool, True, target_sequence, alpha, beta, weight)
            if value < local_value:
                local_value = value
                best_sequence = sequence
            beta = min(beta, local_value)
            if alpha >= beta:
                break

    return best_sequence, local_value

In [15]:
best_gene, best_utility = modified_minimax_search("", modified_nucleotides_pool, True, target_sequence, -math.inf, math.inf, weights)
previous_best_gene, previous_best_utility = minimax_search("", nucleotides_pool, True, target_sequence, -math.inf, math.inf, weights)
if best_utility > previous_best_utility:
    print("YES")
else:  
    print("NO")
print("With special nucleotide")
print(f"Best Gene Sequence Generated: {best_gene},") 
print("Utility Score:", best_utility)

NO
With special nucleotide
Best Gene Sequence Generated: SCTAG,
Utility Score: -96.38
